# 04 — Visualization

**Purpose:** Load saved experiment results and generate interpretable plots.

## What each plot shows

1. **Prediction accuracy (r per channel)** — how well the model predicts each EEG channel.
   Higher bars = brain is more predictable from motion at that location.

2. **TRF kernels** — the brain's temporal response to each feature.
   X-axis = time delay in ms. A peak at 250ms means "the brain responds to that motion
   event 250ms later." Pre-stimulus (negative delays) should be near zero for a valid model.

3. **Experiment comparison** — side-by-side r values across different feature sets.
   Shows which features are actually driving prediction.

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ============================================================
# PRIMARY EXPERIMENT to visualize
EXPERIMENT = 'motion_state_v3'
# ============================================================

data = np.load(f'results/{EXPERIMENT}_results.npz', allow_pickle=True)
kernels   = data['kernels']        # (n_ch, n_feat, n_delays)
r_ch      = data['r_per_channel']  # (n_ch,)
feat_names= list(data['feat_names'])
ch_names  = list(data['ch_names'])
delays_s  = data['delays_s']

print(f'Loaded: {EXPERIMENT}')
print(f'Channels  : {len(ch_names)}')
print(f'Features  : {feat_names}')
print(f'Delay range: {delays_s[0]*1000:.0f} – {delays_s[-1]*1000:.0f} ms')

## Plot 1 — Prediction accuracy per channel

**What the axes mean:**
- **X-axis:** EEG channel name (electrode location on the scalp)
- **Y-axis:** Pearson r — how well the model predicted that channel's EEG signal
  - r = 1.0 would be perfect prediction (never happens)
  - r = 0.0 means the model explains nothing at that channel
  - r = 0.10 is a typical good result for EEG encoding models

**What to expect:** Left frontal (F7, AF3) and temporal (T7, CP5) channels
should be highest — these are language and motor areas. Right and occipital
channels should be near zero.

In [ ]:
order = np.argsort(r_ch)[::-1]
chs_sorted = [ch_names[i] for i in order]
r_sorted   = r_ch[order]

colors = ['#2c7bb6' if r > 0 else '#d7191c' for r in r_sorted]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(chs_sorted)), r_sorted, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(range(len(chs_sorted)))
ax.set_xticklabels(chs_sorted, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Pearson r  (prediction accuracy)', fontsize=11)
ax.set_title(f'TRF prediction accuracy per channel — {EXPERIMENT}', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Annotate best channel
best = order[0]
ax.text(0, r_sorted[0] + 0.003, f'Best: {chs_sorted[0]}\nr={r_sorted[0]:.3f}',
        ha='left', va='bottom', fontsize=9, color='#2c7bb6', fontweight='bold')

plt.tight_layout()
plt.savefig(f'results/{EXPERIMENT}_prediction_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → results/{EXPERIMENT}_prediction_accuracy.png')

## Plot 2 — TRF Kernels (top channels)

**What the axes mean:**
- **X-axis:** Time delay in milliseconds relative to the stimulus event
  - Negative = before the event (pre-stimulus, should be ~flat for a causal response)
  - Zero = the moment the event occurs
  - Positive = after the event (post-stimulus response)
- **Y-axis:** TRF weight — how strongly the brain responds at that delay
  - A peak at 250ms means "the brain's response peaks 250ms after this motion event"
  - Near-zero pre-stimulus weights mean the model is capturing a real response, not noise

**Each panel** is one EEG channel. **Each line** is one feature.

In [ ]:
TOP_N = 6  # how many channels to show
top_idx = np.argsort(r_ch)[::-1][:TOP_N]
delays_ms = delays_s * 1000

# Colors for each feature
feat_colors = plt.cm.tab10(np.linspace(0, 0.9, len(feat_names)))

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=False)
axes = axes.flatten()

for ax_i, ch_i in enumerate(top_idx):
    ax = axes[ax_i]
    for f_i, (fname, color) in enumerate(zip(feat_names, feat_colors)):
        ax.plot(delays_ms, kernels[ch_i, f_i, :], color=color, lw=1.5,
                label=fname if ax_i == 0 else '_')
    ax.axhline(0, color='black', lw=0.6, ls='--')
    ax.axvline(0, color='gray', lw=0.6, ls=':')
    ax.set_title(f'{ch_names[ch_i]}  (r={r_ch[ch_i]:.3f})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Delay (ms)', fontsize=8)
    ax.set_ylabel('Weight', fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.legend(feat_names, loc='upper center', ncol=len(feat_names),
           bbox_to_anchor=(0.5, 1.02), fontsize=8, framealpha=0.8)
fig.suptitle(f'TRF Kernels — top {TOP_N} channels — {EXPERIMENT}',
             fontsize=12, fontweight='bold', y=1.06)
plt.tight_layout()
plt.savefig(f'results/{EXPERIMENT}_kernels.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → results/{EXPERIMENT}_kernels.png')

## Plot 3 — Compare experiments

Load all available experiment results and compare their best-channel r values.

**What to look for:** If a new experiment (e.g. with ELAN+Rokoko regressors)
shows a taller bar than `motion_state_v3`, it means the new features capture
brain activity that raw motion features miss.

In [ ]:
import glob

result_files = sorted(glob.glob('results/*_results.npz'))
if not result_files:
    print('No result files found yet. Run 03_experiments.ipynb first.')
else:
    exp_names, best_r, mean_r = [], [], []
    for fpath in result_files:
        d = np.load(fpath, allow_pickle=True)
        r = d['r_per_channel']
        exp_names.append(os.path.basename(fpath).replace('_results.npz', ''))
        best_r.append(r.max())
        mean_r.append(r[r > 0].mean() if (r > 0).any() else 0)

    x = np.arange(len(exp_names))
    fig, ax = plt.subplots(figsize=(max(8, len(exp_names) * 2), 5))
    bars = ax.bar(x, best_r, color='#2c7bb6', edgecolor='white', linewidth=0.5, label='Best channel r')
    ax.bar(x, mean_r, color='#abd9e9', edgecolor='white', linewidth=0.5, label='Mean r (positive channels)')
    ax.set_xticks(x)
    ax.set_xticklabels(exp_names, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Pearson r', fontsize=11)
    ax.set_title('Experiment comparison — prediction accuracy', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for bar, r in zip(bars, best_r):
        ax.text(bar.get_x() + bar.get_width()/2, r + 0.001, f'{r:.3f}',
                ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.savefig('results/experiment_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → results/experiment_comparison.png')